# Job Match Ranker — LightGBM Training

## Why LightGBM on top of semantic search?
- Cosine similarity captures *semantic* overlap but ignores rate fit, experience level, language, and historical success signals
- LightGBM combines all signals into a single match probability in **< 5ms for 100 jobs**
- Native NaN handling → cold-start freelancers (no history) work out-of-the-box
- After training, inference is a pure C++ tree traversal — no Python overhead per sample

## Homepage flow
```
pgvector cosine search → top-100 jobs
       ↓
LightGBM.predict_proba(features_for_100_jobs)   ← this model
       ↓
ranked top-10 with match %
```

In [3]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
DATA_DIR  = 'content'
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('LightGBM:', lgb.__version__)
print('Setup complete')

LightGBM: 4.6.0
Setup complete


In [4]:
# ── Load data ────────────────────────────────────────────────────────────────
ml_df = pd.read_csv(f'/{DATA_DIR}/ml_training_pairs.csv')
print(f'Loaded {len(ml_df):,} training pairs')
print(f'\nLabel distribution:')
print(ml_df['label'].value_counts())
print(f'\nPositive rate: {ml_df["label"].mean():.3f}')
print(f'\nNull counts (NaN = cold start, expected for performance_score / success_rate_hist):')
print(ml_df.isnull().sum()[ml_df.isnull().sum() > 0])

Loaded 7,143 training pairs

Label distribution:
label
0    6720
1     423
Name: count, dtype: int64

Positive rate: 0.059

Null counts (NaN = cold start, expected for performance_score / success_rate_hist):
performance_score    874
success_rate_hist    874
dtype: int64


In [5]:
# ── Features ────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    # Semantic search output
    'cosine_sim',               # pgvector cosine similarity (0-1)
    # Skill signals
    'skill_overlap_pct',        # % of required skills freelancer has
    'skill_required_matched',   # raw count matched
    'skill_required_total',     # total required skills
    'skill_preferred_pct',      # % preferred skills matched
    # Experience
    'experience_level_match',   # 1 if freelancer >= job requirement
    'exp_delta',                # -2 to +2 (over/under qualified)
    # Budget fit
    'rate_in_budget',           # 1 if rate fits job budget
    'rate_ratio',               # freelancer_monthly / job_budget (capped at 3)
    # Context signals
    'language_match',           # 1 if shares English or Indonesian
    'speciality_match',         # 1 if speciality aligns with job domain
    'domain_match',             # 1 if freelancer domain == job domain
    # Profile richness
    'has_portfolio',            # 1 if has portfolio items
    'work_exp_count',           # number of work experience entries
    # Historical (NaN for cold start — LightGBM handles natively)
    'performance_score',        # overall_performance_score (0-100)
    'success_rate_hist',        # historical success rate (0-100)
    'total_projects',           # total completed projects
    # Cold start flag
    'is_cold_start',            # 1 if no history at all
]

X = ml_df[FEATURE_COLS].copy()
y = ml_df['label'].copy()

print(f'Feature matrix: {X.shape}')
print(f'Features with NaN: {X.isnull().any().sum()} columns (expected: performance_score, success_rate_hist)')

Feature matrix: (7143, 18)
Features with NaN: 2 columns (expected: performance_score, success_rate_hist)


In [6]:
# ── EDA: feature distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

numeric_features = ['cosine_sim', 'skill_overlap_pct', 'skill_preferred_pct',
                     'rate_ratio', 'work_exp_count', 'total_projects',
                     'performance_score', 'success_rate_hist',
                     'experience_level_match', 'rate_in_budget',
                     'domain_match', 'is_cold_start']

for i, feat in enumerate(numeric_features):
    ax = axes[i]
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        subset = X.loc[y == label, feat].dropna()
        ax.hist(subset, bins=20, alpha=0.5, color=color,
                label=f'label={label} (n={len(subset)})', density=True)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Feature distributions by label (red=match, blue=no match)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/feature_distributions.png', dpi=100)
plt.close()
print('Feature distribution plot saved.')

# Correlation with label
corr = X.corrwith(y).abs().sort_values(ascending=False)
print('\nTop feature correlations with label:')
print(corr.to_string())

Feature distribution plot saved.

Top feature correlations with label:
skill_overlap_pct         0.512577
skill_required_matched    0.477706
cosine_sim                0.476862
speciality_match          0.410084
domain_match              0.410084
performance_score         0.133342
success_rate_hist         0.093286
is_cold_start             0.070665
experience_level_match    0.049574
work_exp_count            0.042677
total_projects            0.041915
skill_preferred_pct       0.041552
rate_in_budget            0.039141
exp_delta                 0.035412
rate_ratio                0.013954
has_portfolio             0.013061
language_match            0.012126
skill_required_total      0.006109


In [7]:
# ── Train / val / test split ─────────────────────────────────────────────────
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=42)

print(f'Train:      {len(X_train):,}  (pos={y_train.mean():.3f})')
print(f'Validation: {len(X_val):,}  (pos={y_val.mean():.3f})')
print(f'Test:       {len(X_test):,}  (pos={y_test.mean():.3f})')

Train:      5,160  (pos=0.059)
Validation: 911  (pos=0.059)
Test:       1,072  (pos=0.059)


In [8]:
# ── Baseline: cosine similarity alone ────────────────────────────────────────
# Shows what we get WITHOUT the ML layer — justifies LightGBM
b_cosine   = roc_auc_score(y_test, X_test['cosine_sim'].fillna(0))
b_skill    = roc_auc_score(y_test, X_test['skill_overlap_pct'].fillna(0))
b_combined = roc_auc_score(y_test,
    0.5 * X_test['cosine_sim'].fillna(0) + 0.5 * X_test['skill_overlap_pct'].fillna(0))

print('=== Baseline AUC (no ML, current system) ===')
print(f'  Cosine sim only:        {b_cosine:.4f}')
print(f'  Skill overlap only:     {b_skill:.4f}')
print(f'  0.5*cosine + 0.5*skill: {b_combined:.4f}')
print()
print('LightGBM target: significantly above these numbers')

=== Baseline AUC (no ML, current system) ===
  Cosine sim only:        0.9570
  Skill overlap only:     0.9583
  0.5*cosine + 0.5*skill: 0.9594

LightGBM target: significantly above these numbers


In [9]:
# ── LightGBM Training ────────────────────────────────────────────────────────
# Key design decisions:
#  - class_weight='balanced'  → handles label imbalance automatically
#  - NaN in features          → LightGBM routes NaN to optimal split direction
#  - num_leaves=31, max_depth=6 → fast inference, low memory
#  - early_stopping(50)       → prevents overfitting

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.025,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    class_weight='balanced',
    reg_alpha=0.1,
    reg_lambda=0.2,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)

print(f'Best iteration: {model.best_iteration_}')

[100]	valid_0's binary_logloss: 0.200861
[200]	valid_0's binary_logloss: 0.180826
[300]	valid_0's binary_logloss: 0.171546
[400]	valid_0's binary_logloss: 0.166962
[500]	valid_0's binary_logloss: 0.16323
[600]	valid_0's binary_logloss: 0.162156
Best iteration: 617


In [10]:
# ── Evaluation ───────────────────────────────────────────────────────────────
y_prob = model.predict_proba(X_test)[:, 1]
lgbm_auc = roc_auc_score(y_test, y_prob)
lgbm_ap  = average_precision_score(y_test, y_prob)

print('=== LightGBM Test Results ===')
print(f'  AUC-ROC:           {lgbm_auc:.4f}')
print(f'  Avg Precision:     {lgbm_ap:.4f}')
print(f'  vs cosine baseline: +{lgbm_auc - b_cosine:.4f} AUC')
print(f'  vs combined baseline: +{lgbm_auc - b_combined:.4f} AUC')

# Classification report at threshold optimised for precision-recall balance
threshold = 0.30
y_pred    = (y_prob >= threshold).astype(int)
print(f'\nClassification Report (threshold={threshold}):')
print(classification_report(y_test, y_pred, target_names=['No Match','Match']))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Match','Match'], yticklabels=['No Match','Match'])
ax.set_title(f'Confusion Matrix (threshold={threshold})')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/confusion_matrix.png', dpi=100)
plt.close()
print('Confusion matrix saved.')

=== LightGBM Test Results ===
  AUC-ROC:           0.9679
  Avg Precision:     0.5482
  vs cosine baseline: +0.0109 AUC
  vs combined baseline: +0.0085 AUC

Classification Report (threshold=0.3):
              precision    recall  f1-score   support

    No Match       0.99      0.94      0.96      1009
       Match       0.48      0.92      0.63        63

    accuracy                           0.94      1072
   macro avg       0.74      0.93      0.80      1072
weighted avg       0.96      0.94      0.94      1072

Confusion matrix saved.


In [11]:
# ── Cross-validation ────────────────────────────────────────────────────────
cv_model = lgb.LGBMClassifier(
    n_estimators=model.best_iteration_,
    learning_rate=0.025, max_depth=6, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', n_jobs=-1, random_state=42, verbose=-1,
)
cv_scores = cross_val_score(cv_model, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                             scoring='roc_auc', n_jobs=-1)
print(f'5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Folds: {[f"{s:.4f}" for s in cv_scores]}')

5-Fold CV AUC: 0.9696 ± 0.0060
Folds: ['0.9650', '0.9751', '0.9710', '0.9606', '0.9762']


In [12]:
# ── Cold start analysis ──────────────────────────────────────────────────────
cold_mask = (X_test['is_cold_start'] == 1)
warm_mask = ~cold_mask

print('=== Cold Start vs Warm Start ===')
if cold_mask.sum() > 5:
    cold_auc = roc_auc_score(y_test[cold_mask], y_prob[cold_mask])
    warm_auc = roc_auc_score(y_test[warm_mask], y_prob[warm_mask])
    print(f'  Cold start  n={cold_mask.sum():4d}  AUC={cold_auc:.4f}  (relies on profile features only)')
    print(f'  Warm start  n={warm_mask.sum():4d}  AUC={warm_auc:.4f}  (profile + history)')
    print(f'  Gap: {warm_auc - cold_auc:.4f} — expected, history adds signal')
    print()
    print('Cold start still works because LightGBM uses:')
    print('  cosine_sim, skill_overlap_pct, domain_match, rate_in_budget, has_portfolio')
    print('  NaN features (performance_score, success_rate_hist) are routed to the default split branch')
else:
    print('  Not enough cold-start samples in test set for separate AUC — check cold start ratio in data generation')

=== Cold Start vs Warm Start ===
  Cold start  n=  79  AUC=nan  (relies on profile features only)
  Warm start  n= 993  AUC=0.9652  (profile + history)
  Gap: nan — expected, history adds signal

Cold start still works because LightGBM uses:
  cosine_sim, skill_overlap_pct, domain_match, rate_in_budget, has_portfolio
  NaN features (performance_score, success_rate_hist) are routed to the default split branch


In [13]:
# ── Feature importance ───────────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'gain':       model.booster_.feature_importance(importance_type='gain'),
    'split':      model.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

print('=== Feature Importance (by gain) ===')
print(importance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
top15 = importance_df.head(15)
sns.barplot(data=top15, x='gain', y='feature', palette='viridis', ax=ax)
ax.set_title('Top 15 Features — LightGBM Gain')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/feature_importance.png', dpi=100)
plt.close()
print('Feature importance plot saved.')

=== Feature Importance (by gain) ===
               feature         gain  split
     skill_overlap_pct 70210.672022   1072
            cosine_sim 16884.050336   2163
     performance_score  9282.722947   2070
            rate_ratio  4152.074133   1819
      speciality_match  3867.222144     17
        total_projects  3131.384331   1396
skill_required_matched  2670.647713    625
     success_rate_hist  2587.432713    685
  skill_required_total  1532.297464    652
        work_exp_count  1101.402667    504
             exp_delta   815.209719    362
        language_match   512.897440    181
   skill_preferred_pct   417.515461    208
         has_portfolio   262.416011    136
        rate_in_budget   155.925997     34
experience_level_match    33.631620     23
          domain_match    18.012050      2
         is_cold_start     0.000000      0
Feature importance plot saved.


In [14]:
# ── Inference speed benchmark ────────────────────────────────────────────────
# Simulates: homepage loads, score top-100 pgvector candidates for one freelancer
sample_X = X_test.head(100)

# Warmup (JIT / cache)
for _ in range(10):
    _ = model.predict_proba(sample_X)

N_BENCH = 2000
t0 = time.perf_counter()
for _ in range(N_BENCH):
    scores = model.predict_proba(sample_X)[:, 1]
t1 = time.perf_counter()

ms_per_call = (t1 - t0) / N_BENCH * 1000
us_per_job  = ms_per_call / 100 * 1000

print('=== Inference Speed ===')
print(f'  100-job batch:   {ms_per_call:.3f} ms  (avg over {N_BENCH} runs)')
print(f'  Per job:         {us_per_job:.1f} µs')
print(f'  Page-load safe:  {"YES ✓" if ms_per_call < 50 else "NO — optimise further"}')
print()
print('This runs AFTER pgvector (50-100ms) so total latency = pgvector + lgbm')
print(f'  Estimated total: {50 + ms_per_call:.1f} – {100 + ms_per_call:.1f} ms')

=== Inference Speed ===
  100-job batch:   4.641 ms  (avg over 2000 runs)
  Per job:         46.4 µs
  Page-load safe:  YES ✓

This runs AFTER pgvector (50-100ms) so total latency = pgvector + lgbm
  Estimated total: 54.6 – 104.6 ms


In [15]:
# ── Save model + metadata ────────────────────────────────────────────────────
model_path = f'{MODEL_DIR}/lgbm_job_matcher.pkl'
joblib.dump(model, model_path)
model_kb = os.path.getsize(model_path) / 1024

with open(f'{MODEL_DIR}/feature_cols.json', 'w') as fh:
    json.dump(FEATURE_COLS, fh, indent=2)

summary = {
    'auc_roc':              round(float(lgbm_auc), 4),
    'avg_precision':        round(float(lgbm_ap), 4),
    'baseline_cosine_auc':  round(float(b_cosine), 4),
    'auc_improvement':      round(float(lgbm_auc - b_cosine), 4),
    'cv_auc_mean':          round(float(cv_scores.mean()), 4),
    'cv_auc_std':           round(float(cv_scores.std()), 4),
    'best_iteration':       int(model.best_iteration_),
    'n_features':           len(FEATURE_COLS),
    'n_train':              len(X_train),
    'model_size_kb':        round(model_kb, 1),
    'inference_100_jobs_ms': round(ms_per_call, 3),
}
with open(f'{MODEL_DIR}/model_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f'Model saved: {model_path}  ({model_kb:.1f} KB)')
print(f'Feature list: {MODEL_DIR}/feature_cols.json')
print(f'Summary:      {MODEL_DIR}/model_summary.json')
print()
print('=== FINAL SUMMARY ===')
for k, v in summary.items():
    print(f'  {k:<30s}: {v}')

Model saved: models/lgbm_job_matcher.pkl  (1398.0 KB)
Feature list: models/feature_cols.json
Summary:      models/model_summary.json

=== FINAL SUMMARY ===
  auc_roc                       : 0.9679
  avg_precision                 : 0.5482
  baseline_cosine_auc           : 0.957
  auc_improvement               : 0.0109
  cv_auc_mean                   : 0.9696
  cv_auc_std                    : 0.006
  best_iteration                : 617
  n_features                    : 18
  n_train                       : 5160
  model_size_kb                 : 1398.0
  inference_100_jobs_ms         : 4.641


## Deployment Guide

### Loading the model in FastAPI
```python
import joblib, json, numpy as np

model       = joblib.load('machine_learning/models/lgbm_job_matcher.pkl')
FEAT_COLS   = json.load(open('machine_learning/models/feature_cols.json'))

def rank_jobs(freelancer_features: list[dict]) -> list[float]:
    """freelancer_features: list of feature dicts for each candidate job"""
    X = pd.DataFrame(freelancer_features)[FEAT_COLS]
    return model.predict_proba(X)[:, 1].tolist()
```

### Homepage endpoint flow
```
1. pgvector: SELECT top-100 jobs by cosine similarity
2. For each of the 100 jobs, compute feature dict (skill overlap, exp match, etc.)
3. LightGBM batch predict → 100 match probabilities in < 5ms
4. Sort by probability, return top-10 with match % to frontend
```

### Cold start
- New freelancer with no history: `performance_score=NaN`, `success_rate_hist=NaN`, `total_projects=0`, `is_cold_start=1`
- LightGBM routes NaN to the optimal branch learned during training
- Model still uses: `cosine_sim`, `skill_overlap_pct`, `domain_match`, `rate_in_budget`, `has_portfolio`, `work_exp_count`

### Retraining
Re-run this notebook after collecting ~500 new proposal outcomes (accepted/rejected/completed).
The feedback loop from apply → accept → contract outcome automatically improves ranking quality.

In [17]:
import shutil

zip_filename = 'models_archive'
shutil.make_archive(zip_filename, 'zip', MODEL_DIR)

print(f'Successfully created {zip_filename}.zip containing the models folder for download.')

Successfully created models_archive.zip containing the models folder for download.
